# Phase 2 — VietOCR FT inference on the 1000 private-test images

Loads your `vietocr_ft.pth` (vgg_transformer) and OCRs the **phase-2 private test
set** (`private_test.csv`, ~1000 imgs). Prep (contrast x1.35 + sharpen) + greedy
batched decoding. Auto-adapts: GPU -> MAX_DIM 1280, CPU -> 960.

**Robust to the phase-2 layout** — finds `private_test.csv` and the nested image
folder automatically by image_id (no hard-coded paths).

Outputs to `/kaggle/working/`:
  - `ocr_vietocr_ft_phase2test.parquet`  (download -> I build the product locally)
  - `submission_ocr_only.csv`  (valid format, OCR filled + product blank — to
     sanity-check the 1000-row format on Kaggle)

## Setup
1. New notebook, **GPU T4**, Internet ON.
2. Add Input -> phase-2 dataset (the one with `private_test.csv`).
3. Add Input -> your dataset containing `vietocr_ft.pth`.
4. Run Cell 1 -> **Restart kernel** -> **Save & Run All (Commit)**.

In [ ]:
# Install. Restart kernel after this cell.
!pip install -q easyocr vietocr rapidfuzz
!pip install -q --force-reinstall 'Pillow==10.4.0'
import PIL, torch
print('Pillow', PIL.__version__, '| CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('NOTE: CPU -> slower. For speed: Settings > Accelerator > GPU T4.')

In [ ]:
# ===== phase-2 path discovery + helpers =====
import unicodedata
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm

KIN = Path('/kaggle/input'); WORK = Path('/kaggle/working')

# find the phase-2 test csv (private_test.csv, fallback test.csv)
TEST_CSV = None
for name in ('private_test.csv', 'test.csv'):
    hits = list(KIN.rglob(name))
    if hits:
        TEST_CSV = hits[0]; break
assert TEST_CSV is not None, 'private_test.csv not found - attach the phase-2 dataset'
test_df = pd.read_csv(TEST_CSV, keep_default_na=False)
test_ids = test_df['image_id'].astype(str).tolist()
print('TEST_CSV =', TEST_CSV, '| ids:', len(test_ids))

# sample_submission (for output column order)
SAMPLE = None
for h in KIN.rglob('sample_submission*.csv'):
    SAMPLE = pd.read_csv(h, keep_default_na=False); print('sample_submission =', h); break

# find the image directory by locating the FIRST test id's file anywhere under inputs
def _find_img_dir(sample_id):
    for ext in ('.jpg', '.jpeg', '.png', '.JPG'):
        for p in KIN.rglob(sample_id + ext):
            return p.parent, ext
    return None, None
IMG_DIR, EXT = _find_img_dir(test_ids[0])
assert IMG_DIR is not None, f'could not locate image for id {test_ids[0]}'
print('IMG_DIR =', IMG_DIR, '| ext =', EXT, '| files:', len(list(IMG_DIR.glob('*'+EXT))))

def clean_ocr(t, maxlen=500):
    t = unicodedata.normalize('NFC', str(t)).replace(chr(10),' ').replace(chr(9),' ')
    t = ' '.join(t.split()); toks = t.split()
    ded = [toks[0]] if toks else []
    for w in toks[1:]:
        if w.lower() != ded[-1].lower(): ded.append(w)
    t = ' '.join(ded)
    return t[:maxlen].rstrip() if len(t) > maxlen else t

In [ ]:
# ===== detector + recognizer (greedy batched) + preprocessing =====
import torch, easyocr
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

USE_GPU = torch.cuda.is_available()
DEVICE  = 'cuda:0' if USE_GPU else 'cpu'
MAX_DIM = 1280 if USE_GPU else 960
print(f'device={DEVICE} | MAX_DIM={MAX_DIM}')

WEIGHTS_PATH = None
for p in KIN.rglob('vietocr_ft.pth'):
    WEIGHTS_PATH = str(p); break
if WEIGHTS_PATH is None:
    for p in KIN.rglob('*.pth'):
        if 'ft' in p.name.lower(): WEIGHTS_PATH = str(p); break
assert WEIGHTS_PATH is not None, 'vietocr_ft.pth not found - attach the weights dataset'
print('weights:', WEIGHTS_PATH)

detector = easyocr.Reader(['vi','en'], gpu=USE_GPU, verbose=False)
def detect_boxes(img):
    try:
        horiz, _ = detector.detect(img, min_size=15, text_threshold=0.6)
        return [tuple(int(v) for v in b) for b in (horiz[0] if horiz else [])]
    except Exception:
        return []

ft_cfg = Cfg.load_config_from_name('vgg_transformer')
ft_cfg['device'] = DEVICE
ft_cfg['weights'] = WEIGHTS_PATH
ft_cfg['cnn']['pretrained'] = False
ft_cfg['predictor']['beamsearch'] = False
ft_rec = Predictor(ft_cfg)
print('loaded ft weights (greedy batched)')

def preprocess(bgr, max_dim=MAX_DIM):
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    w, h = img.size
    if max(w, h) > max_dim:
        r = max_dim/max(w, h); img = img.resize((int(w*r), int(h*r)), Image.LANCZOS)
    img = ImageEnhance.Contrast(img).enhance(1.35).filter(ImageFilter.SHARPEN)
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

In [ ]:
# ===== OCR the 1000 phase-2 test images -> parquet + submission.csv =====
EMPTY = {'ocr_text':'','raw_text':'','mean_conf':0.0,'n_boxes':0,'n_chars':0}

def ocr_image(path):
    bgr = cv2.imread(str(path))
    if bgr is None: return dict(EMPTY)
    bgr = preprocess(bgr)
    items = []
    for (x0,x1,y0,y1) in detect_boxes(bgr):
        x0,y0 = max(0,x0), max(0,y0); x1,y1 = min(bgr.shape[1],x1), min(bgr.shape[0],y1)
        if x1-x0 < 6 or y1-y0 < 6: continue
        items.append((y0, x0, Image.fromarray(cv2.cvtColor(bgr[y0:y1, x0:x1], cv2.COLOR_BGR2RGB))))
    if not items: return dict(EMPTY)
    texts = ft_rec.predict_batch([it[2] for it in items])
    ys = [it[0] for it in items]; band = max(8.0, (max(ys)-min(ys))/40.0)
    order = sorted(range(len(items)), key=lambda i: (round(items[i][0]/band), items[i][1]))
    text = clean_ocr(' '.join(texts[i] for i in order if str(texts[i]).strip()))
    return {'ocr_text':text, 'raw_text':text, 'mean_conf':0.0, 'n_boxes':len(items), 'n_chars':len(text)}

OUT = WORK/'ocr_vietocr_ft_phase2test.parquet'
done = {}
if OUT.exists():
    done = {r['image_id']: r for r in pd.read_parquet(OUT).to_dict('records')}
recs = list(done.values()); pend = [i for i in test_ids if i not in done]
print('pending', len(pend), '/', len(test_ids))
for k, iid in enumerate(tqdm(pend)):
    r = ocr_image(IMG_DIR/(iid+EXT)); r['image_id'] = iid; recs.append(r)
    if (k+1) % 100 == 0: pd.DataFrame(recs).to_parquet(OUT, index=False)
ocr = pd.DataFrame(recs); ocr.to_parquet(OUT, index=False)
print('saved', OUT, '| fill', round(float((ocr.ocr_text.str.strip()!='').mean()), 3))

# valid submission (OCR filled, product blank) -> verify 1000-row format on Kaggle
sub = pd.DataFrame({'image_id': test_ids})
sub = sub.merge(ocr[['image_id','ocr_text']], on='image_id', how='left')
sub['ocr_text'] = sub['ocr_text'].fillna('')
sub['product_name'] = ''
sub = sub[['image_id','ocr_text','product_name']]
sub.to_csv(WORK/'submission_ocr_only.csv', index=False, encoding='utf-8')
print('wrote submission_ocr_only.csv | rows', len(sub))
print('\nDONE - download ocr_vietocr_ft_phase2test.parquet; I build the product locally.')